In [0]:
# ================================
# 1. Load Data (Spark → Pandas)
# ================================
df_spark = spark.table("bls_cew.silver.cleaned_data")

# prendre seulement les colonnes utiles (plus rapide)
cols = [
    "year",
    "area_fips",
    "industry_code",
    "own_code",
    "agglvl_code",
    "size_code",
    "sector_type",
    "annual_avg_emplvl",
    "total_annual_wages",
    "avg_annual_pay",
    "covid_period"
]

df = df_spark.select(*cols).toPandas()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ================================
# 2. Basic Info
# ================================
print("\n--- INFO ---")
print(df.info())

print("\n--- DESCRIPTIVE STATS ---")
print(df.describe())


# ================================
# 3. Null Values Check
# ================================
print("\n--- NULL VALUES ---")

nulls = df.isnull().sum()
nulls = nulls[nulls > 0]

if len(nulls) == 0:
    print("Aucune valeur NULL")
else:
    print(nulls)


# ================================
# 4. Duplicates Check
# ================================
print("\n--- DUPLICATES ---")

duplicates = df.duplicated().sum()
print("Nombre de doublons :", duplicates)


# ================================
# 5. Negative Values Check
# ================================
print("\n--- NEGATIVE VALUES ---")

negative_cols = [
    "annual_avg_emplvl",
    "total_annual_wages",
    "avg_annual_pay"
]

for col in negative_cols:
    if col in df.columns:
        count_neg = (df[col] < 0).sum()
        print(f"{col}: {count_neg} valeurs négatives")


# ================================
# 6. Sector Analysis (Public vs Private)
# ================================
print("\n--- SECTOR ANALYSIS ---")

sector_stats = df.groupby("sector_type")[[
    "annual_avg_emplvl",
    "avg_annual_pay"
]].mean().round(2)

print(sector_stats)


# ================================
# 7. COVID Impact Analysis
# ================================
print("\n--- COVID IMPACT ---")

covid_stats = df.groupby("covid_period")[
    "annual_avg_emplvl"
].mean().round(2)

print(covid_stats)


# ================================
# 8. KPI Validation
# ================================
print("\n--- KPI VALIDATION ---")

# éviter division par zéro
df["calculated_avg_pay"] = df.apply(
    lambda row: row["total_annual_wages"] / row["annual_avg_emplvl"]
    if row["annual_avg_emplvl"] and row["annual_avg_emplvl"] != 0 else None,
    axis=1
)

comparison = df[[
    "avg_annual_pay",
    "calculated_avg_pay"
]].dropna().head(10)

print(comparison)

# différence moyenne
diff = (df["avg_annual_pay"] - df["calculated_avg_pay"]).abs().mean()
print("Différence moyenne :", round(diff, 2))


# ================================
# 9. Correlation Analysis
# ================================
print("\n--- CORRELATION ---")

corr = df[[
    "annual_avg_emplvl",
    "total_annual_wages",
    "avg_annual_pay"
]].corr().round(3)

print(corr)


# ================================
# 10. Trend Over Time
# ================================
print("\n--- TREND OVER TIME ---")

trend = df.groupby("year")[
    "annual_avg_emplvl"
].mean().round(2)

print(trend)


# ================================
# 11. Final Validation
# ================================
print("\n--- FINAL VALIDATION ---")

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Duplicates: {duplicates}")
print(f"Avg Pay Difference: {round(diff,2)}")

print("\nValidation completed successfully ")